In [ ]:
# Ref: https://developers.llamaindex.ai/python/framework/integrations/vector_stores/elasticsearchindexdemo/

In [1]:
%pip install -qU llama-index-vector-stores-elasticsearch 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Ollama related
# https://docs.llamaindex.ai/en/stable/examples/embeddings/ollama_embedding/
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import SimpleDirectoryReader

In [8]:
from llama_index.core import Settings

llm = Ollama(
                 model=  "deepseek-r1:7b", #  works best for router agent, # "qwen3.5:4b", #      "llama3.2:latest",
                 request_timeout=3600,  # Change it
                 #temperature = 0.4,
                 context_window=4096,    # Limit timeouts
                 #mirostat = 0
            )

Settings.llm = llm

embed_model = OllamaEmbedding(
                              model_name="nomic-embed-text",      # Using foundational model may be overkill
                              base_url="http://localhost:11434",
                             )

Settings.embed_model = embed_model

In [1]:
from llama_index.vector_stores.elasticsearch import ElasticsearchStore

es = ElasticsearchStore(
    index_name="my_index",
    es_url="http://localhost:9200",
)

In [9]:
from llama_index.core.schema import TextNode

movies = [
    TextNode(
        text="The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.",
        metadata={"title": "Pulp Fiction"},
    ),
    TextNode(
        text="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.",
        metadata={"title": "The Dark Knight"},
    ),
    TextNode(
        text="An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.",
        metadata={"title": "Fight Club"},
    ),
    TextNode(
        text="A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.",
        metadata={"title": "Inception"},
    ),
    TextNode(
        text="A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers.",
        metadata={"title": "The Matrix"},
    ),
    TextNode(
        text="Two detectives, a rookie and a veteran, hunt a serial killer who uses the seven deadly sins as his motives.",
        metadata={"title": "Se7en"},
    ),
    TextNode(
        text="An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
        metadata={"title": "The Godfather", "theme": "Mafia"},
    ),
]

In [3]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.elasticsearch import ElasticsearchStore

In [10]:
def print_results(results):
    for rank, result in enumerate(results, 1):
        print(
            f"{rank}. title={result.metadata['title']} score={result.get_score()} text={result.get_text()}"
        )


In [11]:
def search(
    vector_store: ElasticsearchStore, nodes: list[TextNode], query: str
):
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)

    print(">>> Documents:")
    retriever = index.as_retriever()
    results = retriever.retrieve(query)
    print_results(results)

    print("\n>>> Answer:")
    query_engine = index.as_query_engine()
    response = query_engine.query(query)
    print(response)

In [12]:
from llama_index.vector_stores.elasticsearch import AsyncDenseVectorStrategy

dense_vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",  # for Elastic Cloud authentication see above
    index_name="movies_dense",
    retrieval_strategy=AsyncDenseVectorStrategy(),
)

search(dense_vector_store, movies, "which movie involves dreaming?")

>>> Documents:
1. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.
2. title=Fight Club score=0.0 text=An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.

>>> Answer:
The movie that involves dreaming is **Inception**.


In [13]:
search(dense_vector_store, movies, "which movie involves dreaming?")

>>> Documents:
1. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.
2. title=Fight Club score=0.0 text=An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.

>>> Answer:
The movie that involves dreaming is **Inception**.
